# Brain tumor classifier — Colab all-in-one

This notebook **clones the repo**, **installs dependencies**, **prepares the MRI dataset**, and **runs training** (optional: evaluate-only).

**Before you start**
1. **Runtime → Change runtime type → GPU** (recommended).
2. For **Kaggle** data: create an API token at [kaggle.com/settings](https://www.kaggle.com/settings) — you will upload `kaggle.json` once when prompted.
3. Optional: set a **Hugging Face token** as a Colab secret named `HF_TOKEN` to avoid slow timm downloads (Runtime → Secrets).

**Repository**: change `GIT_BRANCH` or `GITHUB_REPO` in the next cell if you use a fork.


In [ ]:
# --- User settings ---
GITHUB_REPO = "https://github.com/MuhammadSaljooq/Tumor-AI-training-model.git"
GIT_BRANCH = "checkpoint_added"  # use "main" if your fixes are only on main

PROJECT_DIR = "/content/brain_tumor_classifier"
CONFIG_FILE = "configs/config_colab.yaml"

# Models to train: resnet50 | vit | hybrid (space-separated in command)
MODELS = ["hybrid"]  # e.g. ["resnet50"] for a quicker smoke test

# Epochs: None = use config_colab.yaml value; or set an integer to override
EPOCHS_OVERRIDE = None  # e.g. 5

# After the first successful run, set True to skip preprocessing (faster)
SKIP_PREPROCESSING = False

# Where to get raw images: "kaggle" | "drive" | "manual"
DATA_SOURCE = "kaggle"

# Kaggle dataset slug (default matches project README)
KAGGLE_DATASET = "masoudnickparvar/brain-tumor-mri-dataset"

# If DATA_SOURCE == "drive": path to folder that contains Training/ and Testing/
DRIVE_DATA_PATH = "/content/drive/MyDrive/brain_tumor_mri"  # edit to your path

import os
from pathlib import Path
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_SOURCE:", DATA_SOURCE)


## 1) Optional: Hugging Face token (timm / ViT downloads)

If downloads are slow or rate-limited, add a secret `HF_TOKEN` in Colab (Runtime → Secrets) and run:


In [ ]:
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
        print("HF_TOKEN set from Colab secrets.")
except Exception as e:
    print("Secrets/HF_TOKEN not used:", e)


## 2) Check GPU


In [ ]:
!nvidia-smi 2>/dev/null || echo "No nvidia-smi (CPU runtime — training will be slow)."


## 3) Clone repo and install dependencies


In [ ]:
import subprocess
import sys

def run(cmd, cwd=None):
    print("+", " ".join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.check_call(cmd if isinstance(cmd, list) else cmd, shell=True, cwd=cwd)

Path("/content").mkdir(exist_ok=True)
os.chdir("/content")

if Path(PROJECT_DIR).exists():
    run(["rm", "-rf", PROJECT_DIR])

run(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GITHUB_REPO, PROJECT_DIR])
os.chdir(PROJECT_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
print("Ready in:", os.getcwd())


## 4) Dataset → `data/raw`

**Kaggle**: upload `kaggle.json` when prompted (one-time per runtime).

**Drive**: mount Drive and set `DRIVE_DATA_PATH` to the folder containing `Training/` and `Testing/`.

**Manual**: upload a zip or folders into `/content/my_data` and set `MANUAL_DATA_PATH` below.


In [ ]:
MANUAL_DATA_PATH = "/content/my_brain_mri"  # used only if DATA_SOURCE == "manual"

import subprocess
import shutil
from pathlib import Path

def run(cmd, cwd=None):
    print("+", cmd if isinstance(cmd, str) else " ".join(cmd))
    if isinstance(cmd, list):
        subprocess.check_call(cmd, cwd=cwd)
    else:
        subprocess.check_call(cmd, shell=True, cwd=cwd)

def find_training_testing_root(start: Path):
    for training in start.rglob("Training"):
        if training.is_dir():
            parent = training.parent
            if (parent / "Testing").is_dir():
                return parent
    return None

raw_link = Path(PROJECT_DIR) / "data" / "raw"
raw_link.parent.mkdir(parents=True, exist_ok=True)

if raw_link.exists() or raw_link.is_symlink():
    if raw_link.is_symlink():
        raw_link.unlink()
    elif raw_link.is_dir():
        shutil.rmtree(raw_link)

if DATA_SOURCE == "kaggle":
    from google.colab import files
    kdir = Path("/root/.kaggle")
    kdir.mkdir(parents=True, exist_ok=True)
    kfile = kdir / "kaggle.json"
    if not kfile.exists():
        print("Upload your kaggle.json:")
        up = files.upload()
        for name in up:
            Path(name).rename(kfile)
    os.chmod(kfile, 0o600)
    zip_dir = Path("/content/kaggle_brain_mri")
    zip_dir.mkdir(parents=True, exist_ok=True)
    run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(zip_dir)])
    zips = list(zip_dir.glob("*.zip"))
    if not zips:
        raise RuntimeError("No zip downloaded from Kaggle.")
    run(["unzip", "-q", "-o", str(zips[0]), "-d", str(zip_dir / "extracted")])
    root = find_training_testing_root(zip_dir / "extracted")
    if root is None:
        root = zip_dir / "extracted"
        print("Could not find Training+Testing; using extracted root:", root)
    else:
        print("Found split root:", root)
    raw_link.symlink_to(root.resolve(), target_is_directory=True)

elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    src = Path(DRIVE_DATA_PATH)
    if not src.is_dir():
        raise FileNotFoundError(f"DRIVE_DATA_PATH not found: {src}")
    root = find_training_testing_root(src)
    if root is None:
        root = src
    raw_link.symlink_to(root.resolve(), target_is_directory=True)
    print("Linked data/raw ->", raw_link.resolve())

elif DATA_SOURCE == "manual":
    src = Path(MANUAL_DATA_PATH)
    if not src.is_dir():
        raise FileNotFoundError(f"Create {src} and add Training/Testing or class folders, or use kaggle/drive.")
    root = find_training_testing_root(src)
    if root is None:
        root = src
    raw_link.symlink_to(root.resolve(), target_is_directory=True)
    print("Linked data/raw ->", raw_link.resolve())
else:
    raise ValueError("DATA_SOURCE must be kaggle | drive | manual")

print("data/raw points to:", raw_link.resolve())
for child in sorted(raw_link.iterdir())[:8]:
    print(" ", child.name)


## 5) Run training

Uses `configs/config_colab.yaml` (`device: auto`, batch size 16, full `model` block). Checkpoints go to `results/checkpoints/`.


In [ ]:
import subprocess
import sys
import os

os.chdir(PROJECT_DIR)

cmd = [
    sys.executable, "main.py",
    "--config", CONFIG_FILE,
    "--models", *MODELS,
]
if SKIP_PREPROCESSING:
    cmd.append("--skip_preprocessing")
if EPOCHS_OVERRIDE is not None:
    cmd.extend(["--epochs", str(EPOCHS_OVERRIDE)])

print("Running:", " ".join(cmd))
subprocess.check_call(cmd, cwd=PROJECT_DIR)

print("\nCheckpoints:")
ckpt = Path(PROJECT_DIR) / "results" / "checkpoints"
if ckpt.is_dir():
    for f in sorted(ckpt.iterdir()):
        print(f"  {f.name}")
else:
    print("  (no checkpoints dir yet)")


## 6) Optional: evaluate only (after checkpoints exist)

Uncomment and run (set `MODELS` to one model that has a checkpoint):


In [ ]:
# os.chdir(PROJECT_DIR)
# run([sys.executable, "main.py", "--config", CONFIG_FILE, "--models", MODELS[0],
#      "--skip_preprocessing", "--evaluate_only"])
